# Blackjack Environment

- Also known as **21**.
- Consists of a **player** and a **dealer**.
- The player's goal is to get the sum of their card values as close to 21 as possible — either exactly 21, or higher than the dealer's sum — without exceeding 21.

## Card Values

- **Jack (J)**, **King (K)**, and **Queen (Q)** are each worth 10.
- The **Ace (A)** can count as either 1 or 11, whichever benefits the player more.
- Cards 2 through 10 are worth their face value.

## Players and Dealing

- There can be multiple players at a time, but only one dealer.
- All players compete against the dealer individually, not against each other.
- Initially, each player is dealt two cards, both of which are visible to the dealer.
- The dealer is also initially dealt two cards, but shows only one of them to the players.

## Actions

The player can choose between two actions:

- **Hit** — draw one more card.
- **Stand** — take no more cards; this signals the dealer to reveal their remaining card(s).

## Winning Conditions

- Whoever reaches a card sum of exactly 21, or the higher sum without exceeding 21 (compared between player and dealer), wins the game.
- If a player's or dealer's card sum exceeds 21, it is called a **bust**, and that hand loses the game.
- If both the player and the dealer end with the same card sum, the game is a **draw**.

## Usable and Non-Usable Ace

- The player can decide whether an Ace counts as 1 or 11 during play.
- If counting the Ace as 11 helps the player without causing a bust, it is called a **usable Ace**.
- Otherwise (i.e., the Ace must count as 1 to avoid a bust, or counting it as 11 provides no benefit), it is called a **non-usable Ace**.

In [ ]:
import gymnasium as gym
import time

env = gym.make("Blackjack-v1", render_mode="human")
state, info = env.reset()
print(state)
env.render()
time.sleep(10)
env.close()

## Action
- The action stand is represented by 0
- the action hit is represented by 1

## Reward
- **+1.0** reward if we win the game
- **-1.0** reward if we lose the game
- **0** reward if the game is a draw

In [ ]:
print(env.action_space)
print(env.observation_space)

## Every-visit MC prediction with the blackjack game

In [ ]:
import gymnasium as gym
import time
import pandas as pd
from collections import defaultdict

env = gym.make("Blackjack-v1")


# Defining a policy
def policy(state):
    # 0 stand, 1 hit
    # if sum value > 19, then it makes sense to stand
    return 0 if state[0] > 19 else 1


def generate_episode(policy, num_timesteps=100):
    episode = []
    state, info = env.reset()
    print(state)

    for t in range(num_timesteps):
        action = policy(state)
        next_state, reward, done, truncated, info = env.step(action)
        episode.append((state, action, reward))

        if done or truncated:
            print(f'Reached terminal State {next_state} at time step {t + 1}')
            break

        state = next_state

    return episode


def value(policy, num_episodes=500_000, num_timesteps=100):
    total_return = defaultdict(float)
    N = defaultdict(int)

    for i in range(num_episodes):
        episode = generate_episode(policy, num_timesteps)
        print(f"Episode: {i + 1}")

        # Storing all rewards, actions, states obtained from the episode
        states, actions, rewards = zip(*episode)

        # for each step in the episode
        for t, state in enumerate(states):
            # compute return of the state
            R = sum(rewards[t:])
            total_return[state] = total_return[state] + R
            # update number of times state is visited
            N[state] = N[state] + 1

    total_return = pd.DataFrame(total_return.items(), columns=['state', 'total_return'])
    N = pd.DataFrame(N.items(), columns=['state', 'N'])
    df = pd.merge(total_return, N, on="state")

    df['value'] = df['total_return'] / df['N']

    return df

df = value(policy)
print(df.head(10))
time.sleep(5)
env.close()

## First-visit MC prediction with the blackjack game

In [ ]:
import gymnasium as gym
import time
import pandas as pd
from collections import defaultdict

env = gym.make("Blackjack-v1")


# Defining a policy
def policy(state):
    # 0 stand, 1 hit
    # if sum value > 19, then it makes sense to stand
    return 0 if state[0] > 19 else 1


def generate_episode(policy, num_timesteps=100):
    episode = []
    state, info = env.reset()
    print(state)

    for t in range(num_timesteps):
        action = policy(state)
        next_state, reward, done, truncated, info = env.step(action)
        episode.append((state, action, reward))

        if done or truncated:
            print(f'Reached terminal State {next_state} at time step {t + 1}')
            break

        state = next_state

    return episode


def value(policy, num_episodes=500_000, num_timesteps=100):
    total_return = defaultdict(float)
    N = defaultdict(int)

    for i in range(num_episodes):
        episode = generate_episode(policy, num_timesteps)
        print(f"Episode: {i + 1}")

        # Storing all rewards, actions, states obtained from the episode
        states, actions, rewards = zip(*episode)

        # for each step in the episode
        for t, state in enumerate(states):
            if state not in states[0:t]:
                # compute return of the state
                R = sum(rewards[t:])
                total_return[state] = total_return[state] + R
                # update number of times state is visited
                N[state] = N[state] + 1

    total_return = pd.DataFrame(total_return.items(), columns=['state', 'total_return'])
    N = pd.DataFrame(N.items(), columns=['state', 'N'])
    df = pd.merge(total_return, N, on="state")

    df['value'] = df['total_return'] / df['N']

    return df

df = value(policy)
print(df.head(10))
time.sleep(5)
env.close()